# PEFT: LoRA & QLoRA — A Complete Guide

**Parameter-Efficient Fine-Tuning (PEFT)** lets you adapt large pre-trained LLMs to new tasks without retraining all their billions of parameters.

This notebook covers:
1. **LoRA** — Low-Rank Adaptation (small trainable adapters on top of frozen weights)
2. **Quantization** — Compressing model weights to lower precision
3. **QLoRA** — Combining LoRA with 4-bit quantization for extreme memory efficiency

**Key tools:** 🤗 PEFT · `bitsandbytes` · `transformers` · `AutoGPTQ`


---
## 1. The Problem — Why Full Fine-Tuning is Expensive

In a large LLM, weights are the billions of learned numbers (think of them as "knobs" in a control panel). Traditional fine-tuning updates **every single knob**, which requires:
- Storing all parameters in GPU memory (often 32-bit floats)
- Computing and storing gradients for each parameter
- Saving a separate full copy of the model per task

**Example — 7B parameter model:**

| Precision | Memory |
|-----------|--------|
| 32-bit (FP32) | ~28 GB |
| 16-bit (FP16) | ~14 GB |
| 4-bit (QLoRA) | ~3.5 GB ✅ |

PEFT methods solve this by training **only a tiny fraction** of parameters.


---
## 2. LoRA — Low-Rank Adaptation

### Core Idea

Instead of modifying the full weight matrix `W`, LoRA **freezes** the original weights and learns two small **adapter matrices** `A` and `B`:

```
W_new = W_original + B × A
```

Where:
- `A` has shape `(rank × d)` — initialized with small random values
- `B` has shape `(d × rank)` — initialized to **zero** (so training starts neutral)
- `rank` is a small number (typically 4–64), much smaller than `d` (often 1024–4096)
- A scaling factor `α / rank` controls the adaptation strength

### Why Low-Rank Works

Most fine-tuning weight updates have **low intrinsic rank** — meaning the information needed to adapt a model for a new task can be captured with far fewer parameters than the full weight matrix. LoRA exploits this property.

### Parameter Savings

| Matrix Size | Full Update | LoRA (rank=16) | Reduction |
|-------------|-------------|-----------------|-----------|
| 1000 × 1000 | 1,000,000 | 32,000 | **31× fewer** |
| 4096 × 4096 | 16,777,216 | 131,072 | **128× fewer** |

### Analogy
> Instead of rebuilding the entire engine of a car, you bolt on a **turbo booster**. The car (base model) stays unchanged; the booster (LoRA adapter) handles the improvement.


### 2.1 LoRA from Scratch — PyTorch Implementation

In [ ]:
import torch
import torch.nn as nn

# ──────────────────────────────────────────────────────────────────
# Step 1: Visualise the parameter savings
# ──────────────────────────────────────────────────────────────────
d = 1000      # weight matrix dimension (small example)
rank = 16     # LoRA rank

original_params = d * d
lora_params = 2 * rank * d  # A + B

print("=== Parameter Savings ===")
print(f"Original  : {original_params:>10,} parameters  ({original_params * 4 / 1e6:.1f} MB @ 32-bit)")
print(f"LoRA      : {lora_params:>10,} parameters  ({lora_params * 4 / 1e6:.2f} MB @ 32-bit)")
print(f"Reduction : {original_params / lora_params:.1f}×  |  Savings: {(1 - lora_params/original_params)*100:.1f}%")


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 2: Implement a LoRA Linear Layer
# ──────────────────────────────────────────────────────────────────

class LoRALinear(nn.Module):
    """
    Wraps an existing nn.Linear layer with LoRA adapters.

    Forward pass:  output = W_original(x) + scaling * B(A(x))
    Only lora_A and lora_B have requires_grad=True.
    """

    def __init__(self, base_layer: nn.Linear, rank: int = 16, alpha: int = 32):
        super().__init__()
        self.base_layer = base_layer
        self.rank = rank
        self.scaling = alpha / rank          # controls adaptation magnitude

        in_features  = base_layer.in_features
        out_features = base_layer.out_features

        # LoRA matrices
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)

        # Initialisation (same as original LoRA paper)
        nn.init.kaiming_uniform_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)   # zero-init → neutral at start

        # Freeze base layer
        for param in self.base_layer.parameters():
            param.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_layer(x)                          # frozen path
        lora_out = self.lora_B(self.lora_A(x)) * self.scaling  # adapter path
        return base_out + lora_out


# ── Test ──────────────────────────────────────────────────────────
base = nn.Linear(512, 512)
lora_layer = LoRALinear(base, rank=16, alpha=32)

total      = sum(p.numel() for p in lora_layer.parameters())
trainable  = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)

print("=== LoRALinear Parameter Summary ===")
print(f"Total parameters     : {total:,}")
print(f"Trainable (LoRA)     : {trainable:,}  ({trainable/total*100:.2f}%)")
print(f"Frozen (base)        : {total - trainable:,}")

x      = torch.randn(8, 512)
output = lora_layer(x)
print(f"\nInput shape  : {x.shape}")
print(f"Output shape : {output.shape}")


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 3: Demonstrate that rank controls approximation quality
# ──────────────────────────────────────────────────────────────────

def rank_vs_error(size: int = 200, ranks=(1, 4, 16, 32, 64)):
    """Show how different LoRA ranks approximate a random full-rank update."""
    target = torch.randn(size, size) * 0.1   # hypothetical full-rank update

    print(f"{'Rank':<6} {'Parameters':>12}  {'Approximation MSE':>20}")
    print("-" * 42)
    for r in ranks:
        A = torch.randn(r, size) * 0.1
        B = torch.randn(size, r) * 0.1
        approx = B @ A
        mse    = torch.mean((target - approx) ** 2).item()
        params = 2 * r * size
        print(f"{r:<6} {params:>12,}  {mse:>20.6f}")

    print("\n💡 Higher rank → better approximation, but more parameters.")

rank_vs_error()


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 4: A minimal training loop using LoRA
# ──────────────────────────────────────────────────────────────────

def train_lora_demo(rank: int = 4, epochs: int = 100):
    """Train LoRA adapters to learn a simple mapping (input → 2× input)."""
    base      = nn.Linear(10, 10)
    model     = LoRALinear(base, rank=rank, alpha=8)
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=1e-2
    )
    criterion = nn.MSELoss()

    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

    for epoch in range(epochs):
        x      = torch.randn(32, 10)
        target = x * 2.0                    # simple task: double the input

        pred = model(x)
        loss = criterion(pred, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoch {epoch:3d} | Loss: {loss.item():.6f}")

    # Verify
    x_test = torch.ones(1, 10)
    with torch.no_grad():
        out = model(x_test)
    print(f"\nInput  : {x_test.numpy()}")
    print(f"Output : {out.numpy()}")
    print(f"Target : {(x_test * 2).numpy()}")

train_lora_demo()


---
## 3. Quantization

### What Is It?

LLM weights are normally stored as **32-bit floats** (4 bytes each). Quantization maps these to lower-precision representations — fewer bits, less memory, faster inference.

| Format | Bits/weight | Memory (7B model) | Notes |
|--------|-------------|-------------------|-------|
| FP32 | 32 | 28 GB | Training default |
| FP16 / BF16 | 16 | 14 GB | Common for inference |
| INT8 | 8 | 7 GB | Minor quality loss |
| NF4 (4-bit) | 4 | 3.5 GB | Used by QLoRA |

### Analogy
> Weights are like items in a suitcase. Your GPU (bus) can only carry so many. Quantization **compresses the luggage** so more fits on the bus.

### NF4 — The Smart 4-bit Format

Standard 4-bit quantization spaces its 16 levels evenly. **NF4 (NormalFloat4)** spaces them according to the *normal distribution* — more levels near zero, where neural network weights cluster. This gives noticeably better accuracy for the same bit-width.


In [ ]:
import torch
import torch.nn as nn

# ──────────────────────────────────────────────────────────────────
# Step 1: Simple 4-bit quantization demo
# ──────────────────────────────────────────────────────────────────

def simple_quantize(weights: torch.Tensor, bits: int = 4):
    """Uniform quantization: scale to [-2^(bits-1), 2^(bits-1)-1], round, scale back."""
    levels  = 2 ** bits
    half    = levels // 2
    max_val = weights.abs().max()

    if max_val == 0:
        return weights, torch.zeros_like(weights, dtype=torch.int8), 1.0

    scale      = max_val / (half - 1)
    q_ints     = weights.div(scale).round().clamp(-half, half - 1).to(torch.int8)
    q_weights  = q_ints.float() * scale
    return q_weights, q_ints, scale

# Example
w = torch.tensor([0.1234, -0.0987, 0.0456, -0.1111, 0.0789])
q_w, q_ints, scale = simple_quantize(w)

print("=== Simple 4-bit Quantization ===")
print(f"Original   : {w.numpy()}")
print(f"Quantized  : {q_w.numpy()}")
print(f"Int codes  : {q_ints.numpy()}")
print(f"Max error  : {(w - q_w).abs().max():.5f}")
print(f"Memory     : 32-bit → 4-bit  (8× reduction)")


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Step 2: NF4 — quantisation optimised for normal distributions
# ──────────────────────────────────────────────────────────────────

class NF4:
    """
    NormalFloat4 quantization (simplified).
    16 fixed levels designed for zero-mean normal distributions.
    """
    LEVELS = torch.tensor([
        -1.0000, -0.6962, -0.5251, -0.3949,
        -0.2844, -0.1848, -0.0911,  0.0000,
         0.0796,  0.1609,  0.2461,  0.3379,
         0.4407,  0.5626,  0.7230,  1.0000,
    ])

    @classmethod
    def quantize(cls, w: torch.Tensor):
        scale = w.abs().max().clamp(min=1e-8)
        norm  = w / scale                                        # normalise to [-1, 1]
        # find closest NF4 level for each weight
        idx   = (norm.unsqueeze(-1) - cls.LEVELS).abs().argmin(dim=-1)
        q_w   = cls.LEVELS[idx] * scale
        return q_w, idx, scale

    @classmethod
    def dequantize(cls, idx: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
        return cls.LEVELS[idx] * scale


# Compare uniform 4-bit vs NF4 on neural-network-like weights
torch.manual_seed(42)
nn_weights = torch.randn(500) * 0.05      # realistic LLM weight scale

q_uniform, _, _ = simple_quantize(nn_weights, bits=4)
q_nf4, _, _     = NF4.quantize(nn_weights)

err_uniform = (nn_weights - q_uniform).abs().mean()
err_nf4     = (nn_weights - q_nf4).abs().mean()

print("=== Uniform 4-bit vs NF4 ===")
print(f"Uniform 4-bit MAE : {err_uniform:.6f}")
print(f"NF4 MAE           : {err_nf4:.6f}")
print(f"NF4 improvement   : {err_uniform / err_nf4:.2f}× better")


### Using BitsAndBytes for Real 4-bit Quantization

In practice, you use the `bitsandbytes` library via the `transformers` `BitsAndBytesConfig`:

```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # store weights in 4-bit NF4
    bnb_4bit_use_double_quant=True,          # also quantize the scale factors (saves ~0.4 bits/param)
    bnb_4bit_compute_dtype=torch.bfloat16,   # compute in BF16 for speed & stability
    bnb_4bit_quant_type="nf4",               # use NormalFloat4 format
)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    quantization_config=quant_config,
    device_map="auto",
)
```

**Double quantization** further compresses the per-block scale factors themselves (from FP32 to INT8), saving an additional ~0.4 bits per parameter — small but meaningful at 7B+ scale.


---
## 4. QLoRA — Quantized LoRA

QLoRA combines two ideas:

| Component | Role | Precision |
|-----------|------|-----------|
| Base model weights | Frozen; provides pre-trained knowledge | **4-bit NF4** |
| LoRA adapters (A, B) | Trainable; learn task-specific adjustments | **16-bit** |

### How It Works — Step by Step

1. **Load** the base model in 4-bit NF4 (via `bitsandbytes`) → huge memory saving
2. **Freeze** all quantized weights (no gradients)
3. **Attach** small LoRA adapters to selected layers (typically attention projections)
4. **Forward pass**: dequantize → compute → add LoRA output
5. **Backward pass**: gradients flow only through LoRA adapters

### Three Key Innovations (from the original QLoRA paper)
1. **4-bit NF4 quantization** — best quantization format for normally-distributed weights
2. **Double quantization** — quantize the scale factors too (~0.4 bits/param saved)
3. **Paged optimizers** — swap optimizer states to CPU RAM when GPU is full, preventing OOM crashes

### Analogy
> **LoRA** = add a small steering wheel to a luxury bus.  
> **QLoRA** = first make the bus a lightweight model, *then* add the steering wheel.  
> You get almost the same result, but the bus is far cheaper to run.


### 4.1 QLoRA from Scratch — PyTorch Implementation

In [ ]:
import torch
import torch.nn as nn

# Re-use NF4 class from Section 3

class NF4:
    LEVELS = torch.tensor([
        -1.0000, -0.6962, -0.5251, -0.3949,
        -0.2844, -0.1848, -0.0911,  0.0000,
         0.0796,  0.1609,  0.2461,  0.3379,
         0.4407,  0.5626,  0.7230,  1.0000,
    ])

    @classmethod
    def quantize(cls, w):
        scale = w.abs().max().clamp(min=1e-8)
        idx   = (w.div(scale).unsqueeze(-1) - cls.LEVELS).abs().argmin(dim=-1)
        return cls.LEVELS[idx] * scale, idx, scale

    @classmethod
    def dequantize(cls, idx, scale):
        return cls.LEVELS[idx] * scale


# ──────────────────────────────────────────────────────────────────
# Quantized Linear Layer — stores weights in 4-bit
# ──────────────────────────────────────────────────────────────────

class QuantizedLinear(nn.Module):
    """Linear layer whose weights are stored as NF4 indices (4 bits each)."""

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        # Buffers are saved with the model but not trained
        self.register_buffer("weight_idx",   torch.zeros(out_features, in_features, dtype=torch.uint8))
        self.register_buffer("weight_scale", torch.zeros(out_features))

        self._quantize_random_init()

    def _quantize_random_init(self):
        """Initialise with random weights then quantise row-by-row."""
        w = torch.randn(self.out_features, self.in_features) * 0.02
        for i in range(self.out_features):
            _, idx, scale          = NF4.quantize(w[i])
            self.weight_idx[i]    = idx.to(torch.uint8)
            self.weight_scale[i]  = scale

    def dequantized_weight(self) -> torch.Tensor:
        rows = []
        for i in range(self.out_features):
            rows.append(NF4.dequantize(self.weight_idx[i].long(), self.weight_scale[i]))
        return torch.stack(rows)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        W = self.dequantized_weight()
        return nn.functional.linear(x, W)


# ──────────────────────────────────────────────────────────────────
# QLoRA Layer = Quantized base + LoRA adapters
# ──────────────────────────────────────────────────────────────────

class QLoRALinear(nn.Module):
    """
    QLoRA: 4-bit quantized frozen base + 16-bit trainable LoRA adapters.
    """

    def __init__(self, in_features: int, out_features: int, rank: int = 16, alpha: int = 32):
        super().__init__()
        self.base    = QuantizedLinear(in_features, out_features)
        self.scaling = alpha / rank

        # LoRA adapters stay in full 16-bit precision
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)

        nn.init.kaiming_uniform_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)

        # Ensure base is frozen
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x) + self.lora_B(self.lora_A(x)) * self.scaling


# ── Test ──────────────────────────────────────────────────────────
qlora = QLoRALinear(512, 512, rank=16, alpha=32)

total      = sum(p.numel() for p in qlora.parameters())
trainable  = sum(p.numel() for p in qlora.parameters() if p.requires_grad)

print("=== QLoRALinear Parameter Summary ===")
print(f"Total parameters     : {total:,}")
print(f"Trainable (LoRA)     : {trainable:,}  ({trainable/total*100:.2f}%)")
print(f"Frozen (4-bit base)  : {total - trainable:,}")

x = torch.randn(4, 512)
print(f"\nOutput shape         : {qlora(x).shape}")


In [ ]:
# ──────────────────────────────────────────────────────────────────
# Memory comparison: full vs LoRA vs QLoRA (7B model approximation)
# ──────────────────────────────────────────────────────────────────

def memory_breakdown(num_params=7e9, num_layers=32, d_model=4096, lora_rank=16):
    lora_params = num_layers * 4 * d_model * lora_rank * 2   # Q, K, V, O projections

    configs = {
        "Full fine-tune (FP32)"  : (num_params * 4,  num_params),
        "Full fine-tune (FP16)"  : (num_params * 2,  num_params),
        "LoRA  (FP16 base)"      : (num_params * 2 + lora_params * 2, lora_params),
        "QLoRA (4-bit base)"     : (num_params * 0.5 + lora_params * 2, lora_params),
    }

    print(f"{'Method':<25} {'Memory (GB)':>11}  {'Trainable params':>18}  {'Suggested GPU'}")
    print("-" * 80)

    for name, (mem_bytes, train_p) in configs.items():
        mem_gb = mem_bytes / 1e9
        gpu    = ("RTX 3070/4060" if mem_gb <= 8 else
                  "RTX 3080/4070" if mem_gb <= 12 else
                  "RTX 4090/3090" if mem_gb <= 24 else
                  "A100 / H100")
        print(f"{name:<25} {mem_gb:>10.1f}  {int(train_p):>18,}  {gpu}")

memory_breakdown()


In [ ]:
# ──────────────────────────────────────────────────────────────────
# QLoRA training demo — adapters learn a task while base stays frozen
# ──────────────────────────────────────────────────────────────────

def train_qlora_demo(rank=8, epochs=60):
    model     = QLoRALinear(64, 64, rank=rank, alpha=16)
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=1e-2
    )
    criterion = nn.MSELoss()

    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

    for epoch in range(epochs):
        x      = torch.randn(16, 64)
        target = x * 1.5                   # simple task: scale input by 1.5

        loss = criterion(model(x), target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 15 == 0:
            print(f"  Epoch {epoch:3d} | Loss: {loss.item():.6f}")

    # Verify
    x_test = torch.ones(1, 64)
    with torch.no_grad():
        out = model(x_test)
    print(f"\nSample output (expect ~1.5): {out[0, :4].numpy()}")

train_qlora_demo()


---
## 5. Summary & Decision Guide

### How the Three Techniques Compare

| Feature | Full Fine-Tuning | LoRA | QLoRA |
|---------|-----------------|------|-------|
| Base model precision | FP32 / FP16 | FP16 (frozen) | 4-bit NF4 (frozen) |
| Adapter precision | — | FP16 (trained) | FP16 (trained) |
| Trainable parameters | 100% | ~0.1–1% | ~0.1–1% |
| Memory vs full FT | 1× | ~0.5× | ~0.1–0.25× |
| Quality vs full FT | 100% | ~99–100% | ~95–99% |
| Training speed | Slowest | Fast | Slightly slower than LoRA |

### When to Use Each

| Scenario | Recommendation |
|----------|---------------|
| Abundant GPU memory (≥ 40 GB), maximum quality | Full Fine-Tuning |
| 8–16 GB GPU, model < 13 B parameters | **LoRA** |
| 4–12 GB GPU, model ≥ 7 B parameters | **QLoRA** |
| Fine-tuning 70 B models on a single GPU | **QLoRA** |

### Key Takeaways

- **LoRA**: freeze the base model, add two small matrices `A` and `B`. Only train those. Works because weight updates are low-rank.
- **Quantization**: compress weights from 32-bit → 4-bit. NF4 is the best format for neural network weights.
- **QLoRA**: run LoRA on top of a 4-bit quantized base. The base saves memory; the adapters maintain quality.
- Both methods let you **save and swap adapters** (a few MB each) for different tasks on the same base model.

### Ecosystem & Libraries

| Library | Role |
|---------|------|
| 🤗 `peft` | `LoraConfig`, `get_peft_model`, adapter merge/save |
| `bitsandbytes` | 4-bit / 8-bit quantization for `transformers` |
| `transformers` | `BitsAndBytesConfig`, `AutoModelForCausalLM` |
| `trl` | `SFTTrainer` with built-in QLoRA support |
| Axolotl / LLaMA Factory | High-level fine-tuning frameworks |
